# Mask degradation and trace-target sensitivity

Final 2×2 overview. Thin lines are the five subject means after averaging three YNet seeds. The black line and error bars show mean ± sample SD across subjects.

In [ ]:
from pathlib import Path

# Paths work when the notebook is opened either from notebooks/ or the project root.
PROJECT_ROOT = Path('..').resolve()
if not (PROJECT_ROOT / 'src').is_dir():
    PROJECT_ROOT = Path('.').resolve()

MASK_RESULTS = PROJECT_ROOT / 'results/MaskDegeneration/mask_degradation_3seeds_5subjects_clean_metabolite.h5'
TRACE_RESULTS = PROJECT_ROOT / 'results/TraceSweep/trace_sweep_3seeds_5subjects_clean_metabolite.h5'

SHOW_SUBJECT_CURVES = True
FIGSIZE = (8.4, 5.4)

# Set to a Path to save a vector PDF; leave as None to only display the figure.
SAVE_PDF = None  # e.g. PROJECT_ROOT / 'results/MaskDegradation/mask_trace_final.pdf'

In [ ]:
import sys
sys.path.insert(0, str(PROJECT_ROOT / 'src'))

from walinet.paper_figures.mask_degradation.results import (
    aggregates_dataframe as mask_aggregates,
    subject_means_dataframe as mask_subject_means,
)
from walinet.paper_figures.trace_sweep.results import (
    aggregates_dataframe as trace_aggregates,
    subject_means_dataframe as trace_subject_means,
)

for path in (MASK_RESULTS, TRACE_RESULTS):
    if not path.is_file():
        raise FileNotFoundError(path)

mask_summary = mask_aggregates(MASK_RESULTS)
mask_subjects = mask_subject_means(MASK_RESULTS)
trace_summary = trace_aggregates(TRACE_RESULTS)
trace_subjects = trace_subject_means(TRACE_RESULTS)

display(mask_summary)
display(trace_summary)

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import matplotlib as mpl
import matplotlib.ticker as mticker


SAVE_PDF = "SavedGraphics/Figure4_WALINET_sensitivity.pdf"


# ============================================================
# STYLE
# ============================================================

def apply_paper_style():
    plt.rcParams.update({
        "figure.facecolor": "white",
        "axes.facecolor": "white",
        "savefig.facecolor": "white",
        "axes.grid": False,
    })

    mpl.rcParams.update({
        "font.size": 10,
        "axes.titlesize": 11,
        "axes.labelsize": 12,
        "xtick.labelsize": 10,
        "ytick.labelsize": 10,
        "legend.fontsize": 9,
    })


apply_paper_style()


# ============================================================
# SETTINGS
# ============================================================

METRICS = (
    ("residual_l2_over_nuisance_l2", "$\\|\\hat{n}-n\\|_2 / \\|n\\|_2$"),
    ("residual_l2_over_clean_metabolite_l2", "$\\|\\hat{n}-n\\|_2 / \\|m\\|_2$"),
)

COLORS = (
    "#0072B2",   # nuisance
    "#D55E00",   # metabolite
)

LW = 1.8
MS = 3.5
CAPSIZE = 2
ELW = 1.0

TRAIN_TRACE_TARGET = 0.99

FIGSIZE = (8.4, 5.4)


# ============================================================
# HELPERS
# ============================================================

def set_clean_log_ticks(axis, metric_name):

    if metric_name == "residual_l2_over_nuisance_l2":
        ticks = [0.02, 0.03, 0.05, 0.1]

    elif metric_name == "residual_l2_over_clean_metabolite_l2":
        ticks = [0.5, 1.0, 2.0, 5.0]

    else:
        ticks = None

    if ticks is not None:
        axis.set_yticks(ticks)
        axis.yaxis.set_major_formatter(
            mticker.FormatStrFormatter("%g")
        )

    axis.yaxis.set_minor_locator(
        mticker.LogLocator(
            base=10.0,
            subs=(2, 3, 4, 5, 6, 7, 8, 9),
        )
    )

    axis.yaxis.set_minor_formatter(
        mticker.NullFormatter()
    )


# ============================================================
# FIGURE
# ============================================================

figure, axes = plt.subplots(
    2,
    2,
    figsize=FIGSIZE,
)

# Explicit spacing:
# enough vertical space between rows, but still compact
figure.subplots_adjust(
    left=0.10,
    right=0.98,
    bottom=0.12,
    top=0.87,
    wspace=0.27,
    hspace=0.28,
)


# ============================================================
# COLUMN HEADERS
# ============================================================

figure.text(
    0.285,
    0.965,
    "Mask degradation",
    ha="center",
    va="top",
    fontsize=13,
    fontweight="bold",
)

figure.text(
    0.735,
    0.965,
    "Trace-target sensitivity",
    ha="center",
    va="top",
    fontsize=13,
    fontweight="bold",
)


# ============================================================
# LEFT COLUMN: MASK DEGRADATION
# ============================================================

for row_index, ((metric, ylabel), color) in enumerate(
    zip(METRICS, COLORS)
):

    axis = axes[row_index, 0]

    rows = (
        mask_summary[
            mask_summary["metric"] == metric
        ]
        .sort_values("dilation")
    )

    axis.errorbar(
        rows["dilation"],
        rows["mean"],
        yerr=rows["std"],
        color=color,
        linestyle="-",
        marker="o",
        linewidth=LW,
        markersize=MS,
        capsize=CAPSIZE,
        elinewidth=ELW,
        label="Mean ± SD",
    )

    # Original mask
    axis.axvline(
        0,
        color="0.35",
        linestyle="--",
        linewidth=1.0,
        alpha=0.8,
    )

    if row_index == 0:
        axis.set_title("(a) Nuisance-normalized residual")
    else:
        axis.set_title("(c) Metabolite-normalized residual")

    axis.set_ylabel(ylabel)
    axis.set_xticks(rows["dilation"])
    axis.grid(True, alpha=0.2)

    # Only one legend per column
    if row_index == 0:
        axis.legend(
            frameon=False,
            loc="best",
        )

    # Top panel: no x labels/ticks text
    if row_index == 0:
        axis.tick_params(
            axis="x",
            labelbottom=False,
        )

    # Bottom panel gets the shared x-axis label
    else:
        axis.set_xlabel(
            "Mask boundary shift (voxels)"
        )


# ============================================================
# RIGHT COLUMN: TRACE-TARGET SENSITIVITY
# ============================================================

for row_index, ((metric, ylabel), color) in enumerate(
    zip(METRICS, COLORS)
):

    axis = axes[row_index, 1]

    rows = (
        trace_summary[
            trace_summary["metric"] == metric
        ]
        .sort_values("target")
    )

    axis.errorbar(
        rows["target"],
        rows["mean"],
        yerr=rows["std"],
        color=color,
        linestyle="-",
        marker="o",
        linewidth=LW,
        markersize=MS,
        capsize=CAPSIZE,
        elinewidth=ELW,
        label="Mean ± SD",
        zorder=3,
    )

    # Training trace target
    axis.axvline(
        TRAIN_TRACE_TARGET,
        color="0.25",
        linestyle="--",
        linewidth=1.2,
        alpha=0.9,
        label=f"Training target = {TRAIN_TRACE_TARGET:.2f}",
        zorder=2,
    )

    axis.set_yscale("log")
    set_clean_log_ticks(axis, metric)

    if row_index == 0:
        axis.set_title("(b) Nuisance-normalized residual")
    else:
        axis.set_title("(d) Metabolite-normalized residual")

    axis.set_ylabel(ylabel)

    targets = rows["target"].to_numpy()

    axis.set_xticks(targets)

    axis.grid(
        True,
        alpha=0.2,
        which="both",
    )

    # Legend only in top-right panel
    if row_index == 0:
        axis.legend(
            frameon=False,
            loc="best",
        )

    # Top panel: suppress x labels completely
    if row_index == 0:
        axis.tick_params(
            axis="x",
            labelbottom=False,
        )

    # Bottom panel: show x values + axis label
    else:
        axis.set_xticklabels(
            [
                f"{target:.2f}"
                for target in targets
            ],
            rotation=45,
            ha="right",
        )

        axis.set_xlabel(
            "L2-operator trace target"
        )


# ============================================================
# SAVE
# ============================================================

if SAVE_PDF is not None:

    pdf_path = Path(
        SAVE_PDF
    ).expanduser()

    pdf_path.parent.mkdir(
        parents=True,
        exist_ok=True,
    )

    if pdf_path.suffix.lower() != ".pdf":
        pdf_path = pdf_path.with_suffix(".pdf")

    figure.savefig(
        pdf_path,
        format="pdf",
        dpi=300,
        bbox_inches="tight",
    )

    print(
        f"💾 Saved as: {pdf_path.resolve()}"
    )


plt.show()